To select candidates for interviews and to get an overview of edits to the english language wikidata and bot participation, we gathered data from the January 6 United States Capitol Attack (J6USCA) wikidata revision history page.

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime

In [2]:
#January 6 Capitol Attack ID
J6USCA_ID = "Q104705419"
#API
API_URL = "https://www.wikidata.org/w/api.php"

#fetch revision history
def get_edits(item_id):
    edits = []
    rvcontinue = None
    ok = True
    
    while ok:
        params = {"action": "query", "format" : "json", "prop": "revisions", "titles": item_id, "rvlimit": "max", "rvprop": "ids|timestamp|user|comment|flags|size"}
        
        if rvcontinue:
            params["rvcontinue"] = rvcontinue
            
        #retrieve from wikidata
        response = requests.get(API_URL, params = params, headers = {"User-Agent": "GetEditDataResearch"})
        if response.status_code == 429:
            print("sleep 5")
            time.sleep(5)
        response.raise_for_status()
        data = response.json()
        
        pages = data["query"]["pages"]
        
        #get info from all recision history pages
        for page in pages.values():
            edits.extend(page.get("revisions", []))
            
        if "continue" in data:
            rvcontinue = data["continue"]["rvcontinue"]
        else:
            break
            
    return edits

revisions = get_edits(J6USCA_ID)

print(revisions)
        


[{'revid': 2504691802, 'parentid': 2503966829, 'user': 'Vicarage', 'timestamp': '2026-06-11T17:20:01Z', 'size': 167005, 'comment': '/* wbremoveclaims-remove:1| */ [[Property:P625]]: 38°53\'23"N, 77°0\'33"W, QuickStatements 3.0 [[:toollabs:qs-dev/batch/35934|batch #35934]]'}, {'revid': 2503966829, 'parentid': 2503943744, 'user': 'Vicarage', 'timestamp': '2026-06-09T21:04:25Z', 'size': 167520, 'comment': '/* undo:0||2503647430|Quesotiotyo */ Events clearly can have co-ordinates, removing them breaks lots of Wikipedia templates. User refused to engage even when asked repeatedly on talk ([[:toollabs:editgroups/b/EG/9c82e31|details]])'}, {'revid': 2503943744, 'parentid': 2503647430, 'user': 'Pi bot', 'timestamp': '2026-06-09T20:04:46Z', 'size': 167005, 'comment': '/* wbsetclaim-create:2||1 */ [[Property:P625]]: 38°53\'23"N, 77°0\'33"W, Importing coordinate from enwp'}, {'revid': 2503647430, 'parentid': 2500170824, 'user': 'Quesotiotyo', 'timestamp': '2026-06-09T06:11:24Z', 'size': 166474, '

In [3]:
print(len(revisions))
df = pd.DataFrame(revisions)
df.head()

1018


,revid,parentid,user,timestamp,size,comment,anon,minor,commenthidden,userhidden,suppressed
0,2504691802,2503966829,Vicarage,2026-06-11T17:20:01Z,167005,/* wbremoveclaims-remove:1| */ [[Property:P625...,NaN,NaN,NaN,NaN,NaN
1,2503966829,2503943744,Vicarage,2026-06-09T21:04:25Z,167520,/* undo:0||2503647430|Quesotiotyo */ Events cl...,NaN,NaN,NaN,NaN,NaN
2,2503943744,2503647430,Pi bot,2026-06-09T20:04:46Z,167005,/* wbsetclaim-create:2||1 */ [[Property:P625]]...,NaN,NaN,NaN,NaN,NaN
3,2503647430,2500170824,Quesotiotyo,2026-06-09T06:11:24Z,166474,/* wbremoveclaims-remove:1| */ [[Property:P625...,NaN,NaN,NaN,NaN,NaN
4,2500170824,2499912567,Twofivesixbot,2026-05-31T15:55:23Z,167006,/* wbsetclaim-update-qualifiers:1||1|2 */ [[Pr...,NaN,NaN,NaN,NaN,NaN


In [4]:
#see comments to figure out a way to exclude non-en language updates
for index, row in df.iterrows():
    print(row["comment"])

/* wbremoveclaims-remove:1| */ [[Property:P625]]: 38°53'23"N, 77°0'33"W, QuickStatements 3.0 [[:toollabs:qs-dev/batch/35934|batch #35934]]
/* undo:0||2503647430|Quesotiotyo */ Events clearly can have co-ordinates, removing them breaks lots of Wikipedia templates. User refused to engage even when asked repeatedly on talk ([[:toollabs:editgroups/b/EG/9c82e31|details]])
/* wbsetclaim-create:2||1 */ [[Property:P625]]: 38°53'23"N, 77°0'33"W, Importing coordinate from enwp
/* wbremoveclaims-remove:1| */ [[Property:P625]]: 38°53'23"N, 77°0'33"W, #quickstatements; #temporary_batch_1780981630519; property is not to be used on events (these belong to the location (P276) instead)
/* wbsetclaim-update-qualifiers:1||1|2 */ [[Property:P8189]]: 987011253755805171, mv to monolingual text names on J9U statements
/* wbsetdescription-set:1|de */ gewaltsamer Angriff auf das Kapitol durch Anhänger Donald Trumps am 6. Januar 2021
/* wbsetqualifier-add:1| */ [[Property:P9675]]: 131190, Add qualifier to WikiK

In [5]:
#if "label", "alias", "description", "sitelinklink" in comment: look for en/english
wordlist = ["label", "description", "alias", "sitelink"]

#this counts how many comments about labels etc. have no specific language version (possibly multiple languages)
unsure = 0

copied = []

for index, row in df.iterrows():
    
    #skip NaN
    if pd.isna(row["comment"]):
        copied.append(row)
        continue
    
    #include only en-version label/alias/description revisions
    if any(word in row["comment"] for word in wordlist):
        if any(w in row["comment"] for w in ["|en", "English", "english", str("||")]):
            if "||" in row["comment"]:
                unsure +=1
            print("INCLUDE", row["comment"])
            #add this row to new dataframe
            copied.append(row)
        else:
            print("-------------------------- NO ", row["comment"])
    
    else:
        #copy non-label/alias/description revisions
        copied.append(row)
            
print(unsure)

-------------------------- NO  /* wbsetdescription-set:1|de */ gewaltsamer Angriff auf das Kapitol durch Anhänger Donald Trumps am 6. Januar 2021
-------------------------- NO  /* clientsitelink-update:0|shwiki|shwiki:Upad u Kapitol Sjedinjenih Država 6. januara 2021.|shwiki:Napad na Kapitol Sjedinjenih Američkih Država 6. januara */
-------------------------- NO  /* wbsetsitelink-set:1|etwiki */ 2021. aasta rünnak Ameerika Ühendriikide Kapitooliumile
-------------------------- NO  /* wbsetdescription-set:1|ko */ 2021년 1월 6일에 미국 국회의사당에서 대통령 선거인단 투표 집계를 저지하려는 목적으로 발생한 소요사태
-------------------------- NO  /* wbsetlabel-set:1|ko */ 2021년 미국 국회의사당 습격
-------------------------- NO  /* wbsetaliases-add:1|lv */ 6. janvāra uzbrukums ASV Kapitolijam
-------------------------- NO  /* wbsetaliases-add:1|lv */ 2021. gada 6. janvāra uzbrukums ASV Kapitolijam
-------------------------- NO  /* wbsetlabel-add:1|lv */ uzbrukums ASV Kapitolijam
-------------------------- NO  /* wbsetdescription-set:1|pt 

In [6]:
copied_df = pd.DataFrame(copied)

copied_df.head(10)

,revid,parentid,user,timestamp,size,comment,anon,minor,commenthidden,userhidden,suppressed
0,2504691802,2503966829,Vicarage,2026-06-11T17:20:01Z,167005,/* wbremoveclaims-remove:1| */ [[Property:P625...,NaN,NaN,NaN,NaN,NaN
1,2503966829,2503943744,Vicarage,2026-06-09T21:04:25Z,167520,/* undo:0||2503647430|Quesotiotyo */ Events cl...,NaN,NaN,NaN,NaN,NaN
2,2503943744,2503647430,Pi bot,2026-06-09T20:04:46Z,167005,/* wbsetclaim-create:2||1 */ [[Property:P625]]...,NaN,NaN,NaN,NaN,NaN
3,2503647430,2500170824,Quesotiotyo,2026-06-09T06:11:24Z,166474,/* wbremoveclaims-remove:1| */ [[Property:P625...,NaN,NaN,NaN,NaN,NaN
4,2500170824,2499912567,Twofivesixbot,2026-05-31T15:55:23Z,167006,/* wbsetclaim-update-qualifiers:1||1|2 */ [[Pr...,NaN,NaN,NaN,NaN,NaN
6,2483947620,2478517786,VGMaintenanceBot,2026-04-22T17:20:29Z,166369,/* wbsetqualifier-add:1| */ [[Property:P9675]]...,NaN,NaN,NaN,NaN,NaN
14,2460817388,2460816772,Peaceray,2026-02-05T06:35:04Z,165699,/* wbsetclaim-update:2||1|1 */ [[Property:P524...,NaN,NaN,NaN,NaN,NaN
15,2460816772,2460816710,Peaceray,2026-02-05T06:31:05Z,164423,/* wbsetclaim-update:2||1|1 */ [[Property:P803...,NaN,NaN,NaN,NaN,NaN
16,2460816710,2460816545,Peaceray,2026-02-05T06:30:54Z,162016,/* wbsetclaim-update:2||1|1 */ [[Property:P803...,NaN,NaN,NaN,NaN,NaN
17,2460816545,2460816178,Peaceray,2026-02-05T06:30:14Z,159609,/* wbsetclaim-update:2||1|1 */ [[Property:P803...,NaN,NaN,NaN,NaN,NaN


In [7]:
#format timestamp, divide anonymous users and registered

copied_df["timestamp"] = pd.to_datetime(copied_df["timestamp"])
copied_df["minor_edit"] = copied_df.get("minor", pd.Series(False, index=copied_df.index)).notna()
copied_df["anonymous"] = copied_df["user"].str.match(
    r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$",
    na=False
)
copied_df = copied_df.sort_values("timestamp", ascending=True)
display(copied_df.head())

,revid,parentid,user,timestamp,size,comment,anon,minor,commenthidden,userhidden,suppressed,minor_edit,anonymous
1017,1336126283,0,Taavi,2021-01-06 19:49:47+00:00,393,/* wbeditentity-create:2|en */ January 2021 Do...,NaN,NaN,NaN,NaN,NaN,False,False
1016,1336126517,1336126283,Taavi,2021-01-06 19:50:16+00:00,506,/* wbmergeitems-from:0||Q104705421 */ [[MediaW...,NaN,NaN,NaN,NaN,NaN,False,False
1015,1336126684,1336126517,Taavi,2021-01-06 19:50:37+00:00,935,/* wbsetclaim-create:2||1 */ [[Property:P31]]:...,NaN,NaN,NaN,NaN,NaN,False,False
1014,1336126841,1336126684,Taavi,2021-01-06 19:51:02+00:00,1453,/* wbsetclaim-create:2||1 */ [[Property:P585]]...,NaN,NaN,NaN,NaN,NaN,False,False
1013,1336126893,1336126841,Taavi,2021-01-06 19:51:09+00:00,1954,/* wbsetclaim-create:2||1 */ [[Property:P585]]...,NaN,NaN,NaN,NaN,NaN,False,False


In [8]:
#add filtering function
def filter_revisions(
    df,
    user=None,
    start_date=None,
    end_date=None,
    comment_contains=None,
    exclude_bot=None,
    anonymous_only=False,
    registered_only=False
):
    result = df.copy()

    if user:
        result = result[
            result["user"].str.contains(user, case=False, na=False)
        ]

    if start_date:
        result = result[
            result["timestamp"] >= pd.to_datetime(start_date)
        ]

    if end_date:
        result = result[
            result["timestamp"] <= pd.to_datetime(end_date)
        ]

    if comment_contains:
        result = result[
            result["comment"].str.contains(
                comment_contains,
                case=False,
                na=False
            )
        ]
        
    if exclude_bot:
        result = result[
            ~result["user"].str.contains("bot", case=False, na=False)
        ]

    if anonymous_only:
        result = result[result["anonymous"]]
    
    if registered_only:
        result = result[~result["anonymous"]]

    return result

In [9]:
#exclude anonymous users and bots (through username - not a perfect way of doing this, inspired by manual sanity checks of revision history)
non_anon_edits = filter_revisions(
    copied_df,
    registered_only=True,
    exclude_bot=True
)

display(non_anon_edits.head(30))
print(len(non_anon_edits))

,revid,parentid,user,timestamp,size,comment,anon,minor,commenthidden,userhidden,suppressed,minor_edit,anonymous
1017,1336126283,0,Taavi,2021-01-06 19:49:47+00:00,393,/* wbeditentity-create:2|en */ January 2021 Do...,NaN,NaN,NaN,NaN,NaN,False,False
1016,1336126517,1336126283,Taavi,2021-01-06 19:50:16+00:00,506,/* wbmergeitems-from:0||Q104705421 */ [[MediaW...,NaN,NaN,NaN,NaN,NaN,False,False
1015,1336126684,1336126517,Taavi,2021-01-06 19:50:37+00:00,935,/* wbsetclaim-create:2||1 */ [[Property:P31]]:...,NaN,NaN,NaN,NaN,NaN,False,False
1014,1336126841,1336126684,Taavi,2021-01-06 19:51:02+00:00,1453,/* wbsetclaim-create:2||1 */ [[Property:P585]]...,NaN,NaN,NaN,NaN,NaN,False,False
1013,1336126893,1336126841,Taavi,2021-01-06 19:51:09+00:00,1954,/* wbsetclaim-create:2||1 */ [[Property:P585]]...,NaN,NaN,NaN,NaN,NaN,False,False
1011,1336126941,1336126925,Galaktos,2021-01-06 19:51:16+00:00,2505,/* wbsetclaim-create:2||1 */ [[Property:P17]]:...,NaN,NaN,NaN,NaN,NaN,False,False
1010,1336127004,1336126941,Galaktos,2021-01-06 19:51:25+00:00,2928,/* wbsetclaim-create:2||1 */ [[Property:P276]]...,NaN,NaN,NaN,NaN,NaN,False,False
1009,1336127296,1336127004,Taavi,2021-01-06 19:52:08+00:00,3344,/* wbsetclaim-create:2||1 */ [[Property:P276]]...,NaN,NaN,NaN,NaN,NaN,False,False
1008,1336127696,1336127296,Galaktos,2021-01-06 19:53:02+00:00,3781,/* wbsetclaim-create:2||1 */ [[Property:P1269]...,NaN,NaN,NaN,NaN,NaN,False,False
1007,1336127758,1336127696,Taavi,2021-01-06 19:53:11+00:00,4219,/* wbsetclaim-create:2||1 */ [[Property:P361]]...,NaN,NaN,NaN,NaN,NaN,False,False


483


In [10]:
#get unique users
unique_users = non_anon_edits["user"].dropna().unique()

print(f"Total distinct users: {len(unique_users)}\n")

for u in unique_users:
    print(u)

Total distinct users: 119

Taavi
Galaktos
Numberguy6
Zuiarra
Natg 19
J 1982
Thierry Caro
Nintendofan885
Peaceray
Silsha
Marcus Cyron
LennBr
Iconicos
Triggerhippie4
AleatoryPonderings
PtiBzh
CaptainEek
Bravetheif
Evolutionoftheuniverse
KajenCAT
Coffins
Ottawajin
Nachtbold
1857a
Harej
2600:1702:6A0:8540:75DB:CC5:DE88:C9DD
LLs
Superbia23
Koavf
JhowieNitnek
Ctruongngoc
ABPMAB
Rp2006
Paracel63
Kooma
Airon90
Mxn
Wd-Ryan
TnT20052013
Entbert
NMaia
2601:151:C301:33F0:80A0:8994:8580:D1D2
Trivialist
Mannivu
MareBG
Maxlath
2603:7080:1A49:C900:65BF:7C7F:1AC8:575
ViktorQT
Martin Urbanec
Ralf Roletschek
Sänger
Gwennie-nyan
Y2kcrazyjoker4
2601:19B:801:3120:C852:7D9:60BC:2272
Anthony Appleyard
Elli
Rosguill
Aseleste
2601:58C:100:78F0:E58B:49B0:CAD0:8E9B
CmdrDan
Jane023
Belteshassar
Red Slash
Somedifferentstuff
Aoidh
Trygve W Nodeland
Yupik
MelanieN
Pacostein
Geo Swan
2600:1700:8841:D160:780A:795D:B829:A972
Andrew Gray
Kalki
Erik den yngre
TheWxResearcher
Eurohunter
Nivamp
04anotherghost
Ikkingjinnammeb

In [11]:
user_counts = non_anon_edits["user"].value_counts()

print(f"total distinct users: {len(user_counts)}\n")

print(user_counts.head(50))

total distinct users: 119

user
Peaceray                                   69
Triggerhippie4                             54
Thierry Caro                               39
2603:7080:1A49:C900:65BF:7C7F:1AC8:575     28
AleatoryPonderings                         27
2600:1700:8841:D160:780A:795D:B829:A972    21
Nintendofan885                             14
Wd-Ryan                                    11
Ottawajin                                   9
2600:1702:6A0:8540:75DB:CC5:DE88:C9DD       8
Bravetheif                                  7
Taavi                                       7
Numberguy6                                  7
JhowieNitnek                                6
Galaktos                                    6
Yupik                                       6
Nivamp                                      5
Silsha                                      5
KajenCAT                                    4
En Merker                                   4
Geo Swan                                    4
Ko

In [12]:
#sort by time and show number of edits

cutoff = pd.Timestamp("2026-01-10", tz="UTC")
before_cutoff = non_anon_edits[non_anon_edits["timestamp"]<cutoff]
summary = pd.DataFrame({
    "edit_count": before_cutoff["user"].value_counts(),
    "latest_revision": before_cutoff.groupby("user")["timestamp"].max()
})

summary = summary.sort_values(
    "latest_revision",
    ascending=False
)

print(summary.head(50))

                                         edit_count           latest_revision
user                                                                         
Doug Grinbergs                                    3 2026-01-07 05:21:21+00:00
DanGFSouza                                        1 2025-12-24 13:56:47+00:00
UndefinedRachel                                   1 2025-11-14 17:08:59+00:00
Girligaanshub                                     2 2025-09-16 16:02:06+00:00
Trade                                             1 2025-05-25 19:32:51+00:00
Kooma                                             4 2025-01-21 17:23:39+00:00
2A01:9700:9130:3700:C474:5DE3:3B1B:44EC           1 2025-01-20 18:38:23+00:00
2603:7080:29F0:8ED0:E90F:FAA5:1AED:4BA5           1 2024-12-25 18:27:28+00:00
Pyb en résidence                                  1 2024-12-15 23:12:16+00:00
Jock64                                            1 2024-12-13 00:26:18+00:00
Rhododendrites                                    1 2024-10-17 0

We selected a pool of candidates by looking at recent human contributors and contributors with a lot of edits in the past. We verified the users' pages on Wikidata to make sure they were still (somewhat) active. We verified the revision history manually to confirm that their edits were not only administratory tasks but added/removed/modified content of the item page